# Fate of the Universe**Author**: Ricardo Alvim**Date**: January 2026**Purpose**: Paper III - Full cosmological evolution to z < 0---## MethodEvolve Friedmann equations to the far future (z < 0) to determine:- Does acceleration continue? (Big Rip)- Does it transition to deceleration? (Coasting/Big Crunch)- What is the ultimate fate?

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import odeintimport jsonplt.rcParams.update({'font.size': 12, 'figure.dpi': 150})print("="*70)print("FATE OF THE UNIVERSE")print("Full cosmological evolution beyond z = 0")print("="*70)

In [ ]:
# =============================================================# COSMOLOGICAL PARAMETERS# =============================================================H0 = 73.2  # km/s/MpcOmega_m0 = 0.30Omega_de0 = 0.70# Evaporating Universe parameters (from Paper I)w0 = -1.15wa = 0.0  # Constant w for simplicityz_trans = 0.22  # Transition redshiftprint(f"w0 = {w0}, z_trans = {z_trans}")

In [ ]:
# =============================================================# DARK ENERGY EQUATION OF STATE# =============================================================def w_de(z):"""Evaporating Universe w(z):- w = -1 at high z (frozen field)- w < -1 at low z (phantom due to decay)- w approaches -1 + decay_rate in future"""if z > z_trans:return -1.0else:# Smooth transition to phantomdelta_w = w0 - (-1.0)  # = -0.15return -1.0 + delta_w * (1 - z/z_trans)**2 if z >= 0 else w0 + 0.05 * (-z)  # asymptote to w = -1# For future (z < 0): field decays away, w -> -1def w_de_full(z):if z > z_trans:return -1.0elif z >= 0:delta_w = w0 - (-1.0)return -1.0 + delta_w * (1 - z/z_trans)**2else:# Future: phi decays, w asymptotes back to -1return w0 + 0.1 * (1 - np.exp(z))  # z < 0, so this increases w toward -1# Testz_test = np.linspace(-2, 2, 100)w_test = [w_de_full(z) for z in z_test]plt.figure(figsize=(8, 4))plt.plot(z_test, w_test, 'b-', lw=2)plt.axhline(-1, color='gray', ls='--', label='w = -1')plt.axvline(0, color='red', ls=':', label='Today')plt.xlabel('Redshift z')plt.ylabel('w(z)')plt.title('Dark Energy Equation of State')plt.legend()plt.grid(True, alpha=0.3)plt.show()

In [ ]:
# =============================================================# FRIEDMANN EQUATION EVOLUTION# =============================================================def friedmann_ode(y, ln_a):"""Evolve scale factor via Friedmann equation.d(ln H)/d(ln a) = -3/2 * (1 + w_eff)where w_eff = (Omega_m * 0 + Omega_de * w_de) / (Omega_m + Omega_de)"""ln_H = y[0]a = np.exp(ln_a)z = 1/a - 1# Energy densitiesOmega_m = Omega_m0 * a**(-3)# DE with time-varying w# rho_de ~ exp(-3 * integral(1+w) d ln a)# Simplified: use current valuew = w_de_full(z)Omega_de = Omega_de0 * a**(-3*(1+w))# TotalOmega_tot = Omega_m + Omega_de# Effective wif Omega_tot > 0:w_eff = (Omega_m * 0 + Omega_de * w) / Omega_totelse:w_eff = -1# Evolutiond_ln_H = -1.5 * (1 + w_eff)return [d_ln_H]print("Friedmann ODE defined.")

In [ ]:
# =============================================================# SOLVE FROM z=10 TO z=-0.9 (far future)# =============================================================# ln(a) ranges from ln(1/11) to ln(10) = -2.4 to 2.3ln_a_span = np.linspace(-2.4, 2.3, 1000)# Initial condition at z=10: H = H0 * E(10)z_init = 10E_init = np.sqrt(Omega_m0 * (1+z_init)**3 + Omega_de0)ln_H_init = np.log(H0 * E_init)solution = odeint(friedmann_ode, [ln_H_init], ln_a_span)ln_H = solution[:, 0]H = np.exp(ln_H)# Convert to a and za = np.exp(ln_a_span)z = 1/a - 1print(f"Solved from z = {z[0]:.1f} to z = {z[-1]:.2f}")

In [ ]:
# =============================================================# DECELERATION PARAMETER# =============================================================# q = -a * a'' / a'^2 = -1 - d(ln H)/d(ln a)d_ln_H = np.gradient(ln_H, ln_a_span)q = -1 - d_ln_H# Acceleration if q < 0print(f"q at z=0: {q[np.argmin(np.abs(z))]:.3f}")print(f"q at z=-0.5: {q[np.argmin(np.abs(z + 0.5))]:.3f}")

In [ ]:
# =============================================================# FATE DETERMINATION# =============================================================# Check if H remains positive (no Big Crunch)H_future = H[z < 0]H_min = np.min(H_future) if len(H_future) > 0 else H[-1]# Check if acceleration continues (Big Rip) or endsq_future = q[z < 0]q_asymptote = np.mean(q_future[-10:]) if len(q_future) > 10 else q[-1]# Determine fateif H_min <= 0:fate = "Big Crunch"elif q_asymptote < -1:fate = "Big Rip"elif q_asymptote > 0:fate = "Coasting then deceleration"else:fate = "Eternal accelerated expansion (de Sitter)"print(f"\n=== FATE DETERMINATION ===")print(f"H_min (future) = {H_min:.1f} km/s/Mpc")print(f"q asymptote = {q_asymptote:.3f}")print(f"FATE: {fate}")

In [ ]:
# =============================================================# VISUALIZATION# =============================================================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Panel A: Scale factorax = axes[0, 0]ax.semilogy(z, a, 'b-', lw=2)ax.axvline(0, color='red', ls=':', label='Today')ax.set_xlabel('Redshift z')ax.set_ylabel('Scale factor a')ax.set_title('A. Scale Factor Evolution')ax.legend()ax.invert_xaxis()ax.grid(True, alpha=0.3)# Panel B: Hubble parameterax = axes[0, 1]ax.semilogy(z, H, 'g-', lw=2)ax.axhline(H0, color='orange', ls='--', label=f'H0 = {H0}')ax.axvline(0, color='red', ls=':')ax.set_xlabel('Redshift z')ax.set_ylabel('H [km/s/Mpc]')ax.set_title('B. Hubble Parameter Evolution')ax.legend()ax.invert_xaxis()ax.grid(True, alpha=0.3)# Panel C: Deceleration parameterax = axes[1, 0]ax.plot(z, q, 'r-', lw=2)ax.axhline(0, color='gray', ls='--', label='q = 0 (coasting)')ax.axvline(0, color='red', ls=':', label='Today')ax.fill_between(z, -2, 0, where=(q < 0), alpha=0.2, color='green', label='Accelerating')ax.fill_between(z, 0, 2, where=(q > 0), alpha=0.2, color='red', label='Decelerating')ax.set_xlabel('Redshift z')ax.set_ylabel('q (deceleration)')ax.set_title('C. Deceleration Parameter')ax.set_ylim(-2, 1)ax.legend(fontsize=9)ax.invert_xaxis()ax.grid(True, alpha=0.3)# Panel D: Summaryax = axes[1, 1]ax.axis('off')summary = f"""=== FATE OF THE UNIVERSE ===MODEL: Evaporating Universew0 = {w0}z_trans = {z_trans}EVOLUTION:Past (z > 0): Matter dominated -> DE dominatedPresent (z = 0): Accelerating (q = {q[np.argmin(np.abs(z))]:.2f})Future (z < 0): {fate}KEY RESULT:H remains positive (no Big Crunch)q asymptotes to {q_asymptote:.2f}FATE: {fate.upper()}Unlike phantom (Big Rip), the evaporatingfield decays away, avoiding singularities."""ax.text(0.5, 0.5, summary, transform=ax.transAxes, fontsize=10,va='center', ha='center', family='monospace',bbox=dict(facecolor='lightblue', alpha=0.9))plt.suptitle('Fate of the Universe', fontsize=16, fontweight='bold')plt.tight_layout()plt.savefig('fate_of_universe.png', dpi=300)plt.show()

In [ ]:
# =============================================================# SAVE RESULTS# =============================================================results = {"metadata": {"analysis": "Fate of Universe","method": "Friedmann ODE integration"},"parameters": {"w0": float(w0),"z_trans": float(z_trans),"Omega_m0": float(Omega_m0),"Omega_de0": float(Omega_de0)},"results": {"q_today": float(q[np.argmin(np.abs(z))]),"q_asymptote": float(q_asymptote),"H_min_future": float(H_min),"fate": fate},"verdict": "Eternal accelerated expansion without Big Rip","maturity": "Paper Standard","figures": ["fate_of_universe.png"]}with open('fate_of_universe_results.json', 'w') as f:json.dump(results, f, indent=2)print("Saved: fate_of_universe_results.json")try:from google.colab import filesfiles.download('fate_of_universe.png')files.download('fate_of_universe_results.json')print("Downloaded!")except:print("Files saved locally.")